https://colab.research.google.com/drive/1eusTZVhl-njqrufVGZEnebc27za7IaGH#scrollTo=igydy5_qPbNG

https://www.kaggle.com/datasets/nileshmalode1/samsum-dataset-text-summarization

SAMSum Dataset (Samsung Summarization Dataset) là một bộ dữ liệu chuẩn (benchmark) nổi tiếng trong lĩnh vực Xử lý ngôn ngữ tự nhiên (NLP), chuyên dùng cho bài toán Tóm tắt văn bản trừu tượng (Abstractive Text Summarization), đặc biệt là tóm tắt các đoạn hội thoại tin nhắn (chat/dialogue).

1. Nguồn gốc & Mục đích
Tác giả: Được xây dựng và phát triển bởi Samsung R&D Institute Poland.

Mục đích: Các mô hình AI truyền thống thường được huấn luyện trên các văn bản dạng bài báo, tin tức (như CNN/DailyMail). Tuy nhiên, ngôn ngữ trong văn bản chat hàng ngày có nhiều điểm khác biệt: chứa nhiều tiếng lóng, ngữ pháp không chuẩn, mang tính văn nói (informal), ngắt quãng, biểu tượng cảm xúc (emoji) và có sự tham gia của nhiều người. SAMSum ra đời để giải quyết bài toán tóm tắt các cuộc hội thoại thời gian thực như vậy.


Mỗi mẫu dữ liệu trong SAMSum thường bao gồm các trường chính sau:

dialogue: Đoạn hội thoại giữa các nhân vật (thường có từ 2 người trở lên), mô phỏng các tình huống giao tiếp đời thường, công việc, lên lịch hẹn, phàn nàn, v.v.

summary: Bản tóm tắt súc tích, do con người viết (human-annotated), cô đọng lại ý chính của toàn bộ cuộc trò chuyện ở góc nhìn thứ ba.

id: Mã định danh của mẫu dữ liệu đó.

Ví dụ minh họa:

Dialogue:

Alice: Này cậu, tối nay đi ăn lẩu không?
Bob: Tớ bận làm báo cáo đến 8h tối mất rồi.
Charlie: Tớ tham gia được nè! Hay hẹn nhau lúc 8h30 nhé?
Alice: Ok chốt 8h30 quán cũ nha. Bob xong việc chạy ra luôn nhé!
Bob: Oke mọi người.

Summary:

Alice, Bob và Charlie lên kế hoạch đi ăn lẩu lúc 8h30 tối nay. Bob sẽ đến muộn hơn một chút do bận việc.

3. Quy mô & Phân chia (Splits)
Dataset chứa hơn 16.000 đoạn hội thoại bằng tiếng Anh, được chia thành các tập tiêu chuẩn:

Train set: Dùng để huấn luyện mô hình.

Validation set: Dùng để tinh chỉnh siêu tham số và đánh giá trong quá trình train.

Test set: Dùng để đánh giá hiệu năng cuối cùng của mô hình (thường đo bằng các metrics như ROUGE).

4. Thách thức khi xử lý dataset này
Coreference Resolution (Quy chiếu đại từ): AI cần hiểu rõ từ "anh ấy", "cô ấy", "họ" trong đoạn chat đang ám chỉ ai.

Xử lý nhiễu: Các từ viết tắt, tiếng lóng, tin nhắn bị ngắt quãng hoặc đổi chủ đề liên tục.

Abstractive Summarization: Đòi hỏi mô hình phải biết tổng hợp và viết lại ý bằng từ ngữ mới thay vì chỉ đơn thuần sao chép các câu có sẵn trong văn bản (Extractive).

5. Các mô hình thường dùng với SAMSum
Dataset này thường được dùng để fine-tune các mô hình ngôn ngữ lớn hoặc chuyên dụng cho tác vụ tóm tắt như:

PEGASUS (google/pegasus-samsum)

BART (facebook/bart-large-samsum)

Các dòng mô hình mã nguồn mở như T5, FLAN-T5, Llama,..


In [1]:
from transformers import T5ForConditionalGeneration, T5Tokenizer, Trainer, TrainingArguments
import pandas as pd

Câu lệnh này lấy ra 4 thành phần cực kỳ quan trọng từ thư viện transformers của Hugging Face để phục vụ cho việc huấn luyện mô hình ngôn ngữ T5:

T5ForConditionalGeneration:

Ý nghĩa: Kiến trúc mô hình T5 (Text-to-Text Transfer Transformer) dành cho các bài toán sinh văn bản có điều kiện (Conditional Generation).

Tác dụng: Dùng để khởi tạo mô hình T5. Trong bài toán tóm tắt văn bản (Summarization), mô hình sẽ nhận đầu vào là đoạn hội thoại (dialogue) và sinh ra đầu ra là bản tóm tắt (summary).

T5Tokenizer:

Ý nghĩa: Bộ mã hóa/giải mã token của mô hình T5.

Tác dụng: Máy tính không hiểu trực tiếp các từ ngữ con người, nên T5Tokenizer có nhiệm vụ cắt các câu chat thành các mảnh nhỏ gọi là token, sau đó chuyển chúng thành các con số (IDs) để đưa vào mô hình, và ngược lại (chuyển số từ mô hình sinh ra về lại dạng chữ).

Trainer:

Ý nghĩa: Lớp tiện ích (utility class) quản lý toàn bộ quá trình huấn luyện (training) và đánh giá (evaluation) mô hình của Hugging Face.

Tác dụng: Nó giúp tự động hóa việc lặp qua các epoch, tính toán hàm mất mát (loss), thực hiện lan truyền ngược (backpropagation), cập nhật trọng số optimizer và lưu checkpoint mà bạn không cần phải tự viết vòng lặp for thủ công phức tạp.

TrainingArguments:

Ý nghĩa: Lớp cấu hình các tham số huấn luyện.

Tác dụng: Dùng để thiết lập các thông số như: số lượng epoch (num_train_epochs), kích thước batch (per_device_train_batch_size), tốc độ học (learning_rate), thư mục lưu mô hình (output_dir),...

In [2]:
import kagglehub
path = kagglehub.dataset_download("nileshmalode1/samsum-dataset-text-summarization")

100%|██████████| 7.99M/7.99M [00:00<00:00, 33.8MB/s]

Extracting files...


In [3]:
import os

print("Đường dẫn thư mục:", path)
print("Các file có trong thư mục:", os.listdir(path))

Đường dẫn thư mục: /root/.cache/kagglehub/datasets/nileshmalode1/samsum-dataset-text-summarization/versions/1
Các file có trong thư mục: ['samsum-train.csv', 'samsum-test.csv', 'samsum-validation.csv', 'samsum_dataset']


In [7]:
import os
import pandas as pd
import kagglehub

# Lấy đường dẫn thực tế đã tải từ kagglehub thay vì gán cứng đường dẫn Kaggle
path = kagglehub.dataset_download("nileshmalode1/samsum-dataset-text-summarization")
print("Đường dẫn thực tế của dataset:", path)

# 1. Đọc tập Train và Validation
train_data = pd.read_csv(os.path.join(path, "samsum-train.csv"))
validation_data = pd.read_csv(os.path.join(path, "samsum-validation.csv"))

# 2. Xem kích thước dữ liệu
print("Kích thước tập train:", train_data.shape)

# 3. Xem tên các cột
print("Các cột trong file:", train_data.columns.tolist())

# 4. Xem 3 mẫu đầu tiên
for idx, row in train_data.head(3).iterrows():
    print(f"\n--- Mẫu số {idx + 1} ---")
    print(f"ID: {row.get('id', 'N/A')}")
    print(f"Dialogue:\n{row['dialogue']}")
    print(f"Summary:\n{row['summary']}")

Using Colab cache for faster access to the 'samsum-dataset-text-summarization' dataset.
Đường dẫn thực tế của dataset: /kaggle/input/samsum-dataset-text-summarization
Kích thước tập train: (14732, 3)
Các cột trong file: ['id', 'dialogue', 'summary']

--- Mẫu số 1 ---
ID: 13818513
Dialogue:
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)
Summary:
Amanda baked cookies and will bring Jerry some tomorrow.

--- Mẫu số 2 ---
ID: 13728867
Dialogue:
Olivia: Who are you voting for in this election? 
Oliver: Liberals as always.
Olivia: Me too!!
Oliver: Great
Summary:
Olivia and Olivier are voting for liberals in this election. 

--- Mẫu số 3 ---
ID: 13681000
Dialogue:
Tim: Hi, what's up?
Kim: Bad mood tbh, I was going to do lots of stuff but ended up procrastinating
Tim: What did you plan on doing?
Kim: Oh you know, uni stuff and unfucking my room
Kim: Maybe tomorrow I'll move my ass and do everything
Kim: We were going to defrost a fridge so instead 

In [8]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [9]:
train_data.shape

(14732, 3)

In [10]:
train_data = train_data.sample(n=6000, random_state=42).reset_index(drop=True)
validation_data = validation_data.sample(n=500, random_state=42).reset_index(drop=True)

In [11]:
display(train_data.shape)
display(validation_data.shape)

(6000, 3)

(500, 3)

# Load Datasets

# Data Preprocessing

In [12]:
train_data['dialogue'][0]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting\r\nViolet: <file_other>\r\nClaire: Hi! :) Thanks, but I've already read it. :)\r\nClaire: But thanks for thinking about me :)"

In [13]:
#Clean the text by removing unwanted characters
import re

def clean_text(text):
  if not isinstance(text, str):
    return ""
  text = re.sub(r'\r\n', ' ', text) #remove carriage and newline space
  text = re.sub(r'\s+', ' ', text) #remove extra space but keep single space
  text = re.sub(r'<.*?>', '', text) #remove html tags
  text = text.strip().lower() #Added parentheses here

  return text

# Đọc lại dữ liệu gốc để tránh xung đột với các dữ liệu lỗi đã bị gán nhầm trước đó
train_data = pd.read_csv(os.path.join(path, "samsum-train.csv"))
validation_data = pd.read_csv(os.path.join(path, "samsum-validation.csv"))

# Lấy mẫu lại
train_data = train_data.sample(n=6000, random_state=42).reset_index(drop=True)
validation_data = validation_data.sample(n=500, random_state=42).reset_index(drop=True)

#apply cleaning to dialogue and summary columns
train_data['dialogue'] = train_data['dialogue'].apply(clean_text)
train_data['summary'] = train_data['summary'].apply(clean_text)

validation_data['dialogue'] = validation_data['dialogue'].apply(clean_text)
validation_data['summary'] = validation_data['summary'].apply(clean_text)

#display a sample after cleaning
train_data.head()

,id,dialogue,summary
0,13811908,violet: hi! i came across this austin's articl...,violet sent claire austin's article.
1,13716431,pat: so does anyone know when the stream is go...,pat and lou are waiting for the stream but kev...
2,13810214,jane: jane: whaddya think? shona: this ur tin...,jane is updating her tinder profile tonight an...
3,13729823,"adam: do u have a map of paris? tom: yes, why?...",tom has a map of paris.
4,13681400,"frank: hi, how's the family? mike: great! sam'...","mike is happy, because sam's moved out. mike a..."


# Tokenization

In [14]:
tokenizer = T5Tokenizer.from_pretrained('t5-small')

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Đây là lớp xử lý văn bản (Tokenizer) được thiết kế riêng cho dòng mô hình T5 (Text-to-Text Transfer Transformer) của Google.

T5Tokenizer có nhiệm vụ:

Cắt nhỏ văn bản (Tokenization): Chia câu thành các mảnh nhỏ hơn gọi là token (có thể là một từ, một phần của từ, hoặc ký tự).

Chuyển đổi hai chiều (Encode/Decode): Biến các chuỗi văn bản thành các mã số nguyên (input_ids) để đưa vào mô hình AI, và ngược lại, dịch các con số mà mô hình sinh ra ở đầu ra thành chữ viết cho con người đọc được.


.from_pretrained('t5-small')
Ý nghĩa: Phương thức tĩnh giúp tải xuống và khởi tạo bộ từ điển (vocabulary) cùng các quy tắc cắt token được định nghĩa sẵn từ kho lưu trữ mã nguồn mở Hugging Face Hub.

't5-small': Là tên phiên bản mô hình T5 nhỏ nhất (có khoảng 60 triệu tham số).

Khi bạn truyền chuỗi này, T5Tokenizer sẽ tải đúng bộ quy tắc tách từ, các ký tự đặc biệt (như <pad>, </s>), và từ điển tương ứng mà mô hình t5-small đã được huấn luyện trước đó. Điều này đảm bảo cách cắt chữ của bạn lúc đưa vào huấn luyện hoặc dự đoán khớp hoàn toàn với cấu trúc của mô hình.

In [15]:
#preprocessing function for tokennization
def preprocess_function(examples):
  inputs = tokenizer(examples['dialogue'], padding='max_length', truncation=True, max_length=512)
  targets = tokenizer(examples['summary'], padding='max_length', truncation=True, max_length=150)

  inputs['labels'] = targets['input_ids']

  return inputs


dùng để chuẩn bị dữ liệu (tiền xử lý) cho các mô hình dạng Seq2Seq như T5 trước khi đưa vào quá trình huấn luyện.

inputs = tokenizer(examples['dialogue'], padding='max_length', truncation=True, max_length=512)
Ý nghĩa: Mã hóa phần nội dung đoạn hội thoại (dialogue) — đây là phần đầu vào (input) của mô hình.

Các tham số:

padding='max_length': Đệm thêm các token rỗng vào câu nếu độ dài ngắn hơn mức cho phép, đảm bảo tất cả các mẫu trong batch đều có chung một độ dài cố định.

truncation=True: Cắt bớt câu nếu độ dài vượt quá giới hạn cho phép.

max_length=512: Giới hạn tối đa là 512 token cho một đoạn hội thoại đầu vào.


targets = tokenizer(examples['summary'], padding='max_length', truncation=True, max_length=150)
Ý nghĩa: Mã hóa phần bản tóm tắt (summary) — đây là phần đầu ra mục tiêu (target/label) mà mô hình cần học cách sinh ra.

Các tham số: Giống như trên nhưng giới hạn ở max_length=150 token vì bản tóm tắt thường ngắn hơn đoạn hội thoại gốc.

nputs['labels'] = targets['input_ids']
Ý nghĩa: Gán các mã token của phần summary vào khóa 'labels' bên trong từ điển inputs.

Tác dụng: Trong thư viện Hugging Face, các mô hình kiến trúc Seq2Seq (như T5, BART) khi huấn luyện bằng Trainer bắt buộc phải nhận nhãn (ground truth) thông qua trường labels để so sánh với kết quả dự đoán nhằm tính toán độ sai số (loss).

In [16]:
#apply the preprecessing
train_dataset = train_data.apply(preprocess_function, axis=1)
validation_dataset = validation_data.apply(preprocess_function, axis=1)

để tiền xử lý dữ liệu

In [17]:
train_data['dialogue'][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:  claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

In [18]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

input_ids: Danh sách các con số đại diện cho các từ/token của đoạn hội thoại (dialogue) sau khi đã cắt nghĩa qua từ điển của Tokenizer và được đệm (padding) đủ độ dài 512.

attention_mask: Một mảng gồm các số 1 và 0 (có độ dài tương ứng với input_ids):

Số 1 tương ứng với vị trí có từ thật.

Số 0 tương ứng với các vị trí là khoảng trắng đệm thêm (<pad>). Điều này giúp mô hình biết vị trí nào cần tập trung chú ý và vị trí nào cần bỏ qua.

labels: Danh sách các con số đại diện cho phần tóm tắt (summary) đã được đệm đủ độ dài 150, dùng làm đáp án chuẩn để mô hình đối chiếu trong quá trình học.

In [19]:
max([len(text) for text in train_data['dialogue']])

4143

Tính độ dài tính theo số lượng ký tự (characters) của đoạn hội thoại dài nhất trong tập dữ liệu.



In [20]:
max([len(tokenizer.encode(text)) for text in train_data['dialogue']])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (539 > 512). Running this sequence through the model will result in indexing errors


1224

Tính độ dài tính theo số lượng token sau khi đã được cắt qua bộ mã hóa (tokenizer) của mô hình.

tokenizer.encode(text) sẽ băm nhỏ chuỗi văn bản thành các token (mỗi từ hoặc một phần của từ tương ứng với mã số trong từ điển của mô hình).

Ký tự (len(text) = 4143) tính đếm từng chữ cái, dấu cách, dấu phẩy, dấu chấm, dấu xuống dòng \n.

Token (tokenizer.encode) là các đơn vị từ hoặc một phần của từ (sub-word) trong từ điển của mô hình.

# Fine Tuning Model

In [21]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

dùng để tải và khởi tạo mô hình AI từ kho lưu trữ của Hugging Face.

T5ForConditionalGeneration
Ý nghĩa: Đây là lớp mô hình chuyên dụng trong thư viện transformers dành cho các tác vụ sinh văn bản có điều kiện (Conditional Text Generation) dựa trên kiến trúc T5 của Google.

Tác dụng: Nó bao gồm toàn bộ cấu trúc mạng neural (bao gồm cả phần Encoder để đọc hiểu văn bản đầu vào và Decoder để sinh ra văn bản đầu ra). Các bài toán như tóm tắt văn bản (summarization), dịch máy, hoặc trả lời câu hỏi đều sử dụng lớp mô hình này.

.from_pretrained('t5-small')
Ý nghĩa: Phương thức giúp tải xuống các trọng số (weights) đã được huấn luyện sẵn từ trước (pre-trained weights) từ Hugging Face Hub.

't5-small': Là tên phiên bản mô hình T5 nhỏ nhất (có khoảng 60 triệu tham số).

Khi bạn chạy dòng code này, máy tính sẽ tải trọng số của mô hình t5-small về máy (hoặc dùng bản đã cache) để bạn có thể tiếp tục đem đi fine-tune (huấn luyện tinh chỉnh) trên tập dữ liệu SAMSum của mình.


In [22]:
# Cấu hình các tham số và thiết lập cho quá trình huấn luyện mô hình
training_args = TrainingArguments(
    output_dir="./results",  # Thư mục lưu các checkpoint và kết quả huấn luyện sau mỗi bước/epoch
    num_train_epochs=3,  # Giảm số epochs xuống 3 để chạy nhanh hơn trên Colab
    per_device_train_batch_size=4,  # Giảm batch size xuống 4 để tránh tràn bộ nhớ GPU (OOM)
    per_device_eval_batch_size=4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=10,  # Tần suất ghi log sau mỗi 10 bước
    eval_strategy="epoch",  # Thời điểm đánh giá: sau mỗi epoch
    report_to="none"  # Không đẩy log lên wandb
)

# Chuyển đổi Pandas Series chứa dicts thành một List của các dicts hợp lệ cho Trainer
train_list = train_dataset.tolist()
validation_list = validation_dataset.tolist()

# Khởi tạo đối tượng Trainer
trainer = Trainer(
    model=model,  # Mô hình T5
    args=training_args,  # Tham số huấn luyện đã sửa đổi
    train_dataset=train_list,  # Tập huấn luyện định dạng list
    eval_dataset=validation_list,  # Tập đánh giá định dạng list
)

# Bắt đầu chạy quá trình huấn luyện mô hình
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.360188,0.354057
2,0.326113,0.346728
3,0.360059,0.345373


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4500, training_loss=0.5558606966336568, metrics={'train_runtime': 1253.4274, 'train_samples_per_second': 14.361, 'train_steps_per_second': 3.59, 'total_flos': 2436152426496000.0, 'train_loss': 0.5558606966336568, 'epoch': 3.0})

4500/4500: Tổng số bước (steps) huấn luyện đã hoàn thành là 4500 bước (tương ứng với toàn bộ lượng dữ liệu trong tập train được duyệt qua trong 3 epoch).

20:50: Tổng thời gian huấn luyện mất khoảng 20 phút 50 giây (trên phần cứng GPU bạn đang dùng như Google Colab).

Epoch 3/3: Đã chạy đủ toàn bộ 3 vòng (epochs) như cấu hình đã đặt.

training_loss: 0.5558: Giá trị loss trung bình tính trên toàn bộ quá trình train.

train_runtime: 1253.4s: Tổng thời gian chính xác tính bằng giây (~20 phút 53 giây).

train_samples_per_second: 14.36: Trung bình mỗi giây mô hình xử lý được khoảng 14.36 mẫu hội thoại.

# Save and load model


In [23]:
model.save_pretrained('./saved_summary_model')
tokenizer.save_pretrained('./saved_summary_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [24]:
#load the savved model and tokenizer
model = T5ForConditionalGeneration.from_pretrained('./saved_summary_model')
tokenizer = T5Tokenizer.from_pretrained('./saved_summary_model')

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# Summarization System

In [32]:
# Định nghĩa lại hàm summarize_dialogue sau khi sửa lỗi biến 'values' -> 'value'
device = model.device

def summarize_dialogue(dialogue):
  dialogue = clean_text(dialogue)
  inputs = tokenizer(dialogue, return_tensors='pt', max_length=512, truncation=True)

  # Đã sửa lỗi: key, value (dùng thống nhất số ít)
  inputs = {key: value.to(device) for key, value in inputs.items()}

  outputs = model.generate(
    inputs['input_ids'],
    max_length=150,
    num_beams=4,
    early_stopping=True
  )

  summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
  return summary


In [33]:

# Thử nghiệm với dữ liệu mẫu đầu vào
sample_dialogue = """
Violet: Hey Claire! I was reading an article about Austin and thought you might find it interesting.
Violet: It's about the current trends in urban development and how cities are planning for growth.
Violet: Here, let me share the link: <file_other>
Claire: Oh wow, that sounds like an insightful read. But I've actually already read that one.
Claire: It was really interesting though, especially the part about sustainable architecture.
Claire: You know, I've been following these urban planning discussions for a while now.
Violet: Oh, I didn't know that! Well, I'll look for something else then, maybe something else.
Claire: That would be awesome! Let me know if you find something cool.
Violet: Sure, I'll keep you posted. Thanks for the feedback!
"""

summary = summarize_dialogue(sample_dialogue)
print(summary)

claire was reading an article about austin. it's about the current trends in urban development and how cities are planning for growth.


# Download to machine

In [30]:
import shutil
#Path to the diretory containing the fine-tuned model
model_dir = "./saved_summary_model"

#Output zip file path
output_zip_path = "saved_summary_model.zip"

#Create a zip archive
shutil.make_archive(base_name="saved_summary_model", format="zip", root_dir=model_dir)

'/content/saved_summary_model.zip'

In [31]:
from IPython.display import FileLink

#Display a download link
FileLink(output_zip_path)

/content/saved_summary_model.zip

In [34]:
import gradio as gr
from transformers import T5ForConditionalGeneration, T5Tokenizer

# 1. Tải mô hình và tokenizer từ thư mục đã lưu trước đó
model_path = "./saved_summary_model"
print("Đang tải mô hình từ thư mục:", model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

# Xác định thiết bị (GPU nếu có, không thì dùng CPU)
device = model.device


# 2. Định nghĩa hàm xử lý tóm tắt hội thoại
def summarize_dialogue(dialogue):
    if not dialogue.strip():
        return "Vui lòng nhập nội dung đoạn hội thoại!"

    # Tiền xử lý (nếu bạn có hàm clean_text thì gọi ở đây, nếu không có thể bỏ qua)
    # dialogue = clean_text(dialogue)

    # Tokenize đoạn hội thoại
    inputs = tokenizer(
        dialogue, return_tensors="pt", max_length=512, truncation=True
    )

    # Chuyển tensor sang thiết bị tương ứng của model (GPU/CPU)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Cho mô hình sinh tóm tắt
    outputs = model.generate(
        inputs["input_ids"],
        max_length=150,
        min_length=20,
        num_beams=4,
        early_stopping=True,
    )

    # Giải mã token thành văn bản kết quả
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary


# 3. Xây dựng giao diện Web bằng Gradio
demo = gr.Interface(
    fn=summarize_dialogue,
    inputs=gr.Textbox(
        lines=10,
        placeholder="Dán đoạn hội thoại chat vào đây (Ví dụ: Violet: Hello...)",
        label="Đoạn hội thoại gốc (Dialogue)",
    ),
    outputs=gr.Textbox(label="Bản tóm tắt do mô hình sinh ra (Summary)"),
    title="🤖 Ứng dụng Tóm tắt Đoạn hội thoại với T5 (SAMSum)",
    description=(
        "Nhập hoặc dán nội dung chat vào ô bên trái, sau đó bấm nút **Submit**"
        " để mô hình T5 đã fine-tune tự động tóm tắt nội dung chính."
    ),
    examples=[
        [
            (
                "Violet: Hey Claire! I was reading an article about Austin and"
                " thought you might find it interesting.\nViolet: It's about"
                " the current trends in urban development and how cities are"
                " planning for growth.\nViolet: Here, let me share the link:"
                " <file_other>\nClaire: Oh wow, that sounds like an insightful"
                " read. But I've actually already read that one.\nClaire: It"
                " was really interesting though, especially the part about"
                " sustainable architecture.\nClaire: You know, I've been"
                " following these urban planning discussions for a while"
                " now.\nViolet: Oh, I didn't know that! Well, I'll look for"
                " something else then, maybe something else.\nClaire: That"
                " would be awesome! Let me know if you find something"
                " cool.\nViolet: Sure, I'll keep you posted. Thanks for the"
                " feedback!"
            )
        ]
    ],
)

# 4. Chạy ứng dụng web
if __name__ == "__main__":
    demo.launch(
        share=True
    )  # share=True sẽ tạo một đường dẫn công khai tạm thời để bạn truy cập từ xa hoặc chia sẻ

Đang tải mô hình từ thư mục: ./saved_summary_model


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d9eb66f2dd7b84eccd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
